**--------------------**<br>
**TALE WRITER**<br>
**--------------------**<br><br>
Welcome to TALE Writer! A Python-based design tool for the efficient design of TALE repeat arrays. This script was made for the FusX TALE Base Editor (FusXTBE) project in the Precision Gene Editing Laboratory of Dr. Stephen C. Ekker at the Mayo Clinic by Ph.D candidate Santiago Restrepo-Castillo. The FusXTBE technology is based on the FusX assembly system, and on the first-ever mitochondrial base editor, DdCBE, which was developed by the Liu Group.<br><br>
Broadly, TALE Writer can be used to design TALENs or FusXTBEs against a specific target sequence. In particular, FusXTBE design allows mapping all potential 5'-TC-3' target sites for C-to-T base editing in a given sequence. Additionally, FusXTBEs can be specifically designed for premature termination codon (PTC) induction (a DNA edit resulting in the generation of a non-functional gene product due to the premature termination of translation). For this end, the script looks for the following target motifs within any given sequence:<br>
* The 5’-TGA-3’ motif (TGA in -frame), in which a G-to-A edit results in the stop codon UAA.<br>
* The 5’-TCAA-3’ motif (CAA in-frame), in which a C-to-T edit results in the stop codon UAA.<br>
* The 5’-TCAG-3’ motif (CAG in-frame), in which a C-to-T edit results in the stop codon UAG.<br>

**USER-DEFINED INPUT:**<br>
* **TALENs:**<br><br>
  * **Use default design parameters?** Type 'no' to define custom design parameters or 'yes' to proceed.<br>
    * All default minima and maxima are 14 and 16 bp, respectively. Custom limits are between 10 and 25 bp.<br>
  * **Sequence (5'-to-3'):** Input sequence containing the cut site region.<br>
  * **Target coordinate:** The coordinate within the input sequence that corresponds to the desired cut site.<br>
  * **Save results as CSV file?** Type 'yes' to save the results or 'no' to proceed.<br>
    * **Alphanumeric filename:** Type an alphanumeric filename to save the results<br>
  * **New sequence?** Type 'yes' to try another sequence or 'no' to exit.<br><br>
* **FusXTBEs:**<br><br>
  * **5'-TC-3' or PTC induction?** Choose whether to design base editors against all 5'-TC-3' motifs within a given sequence or for PTC induction.
  * **Use default design parameters?** Type 'no' to define custom design parameters or 'yes' to proceed.<br>
    * Default parameters and limits for custom parameters regarding arms and spacer length are the same as for TALEN design.
    * **Target window**: The script looks for all potential target sites within the protospacer region at least 'x' bp from the target sequence of either TALE repeat array, where 'x' is the target window. For example, if the length of the spacer is 16 bp and the target window is 4 bp, target Cs/Gs are searched from the fifth to the twelfth bp within the protospacer. The default target window is 4 bp. Custom target windows cannot be less than 0 or greater than half the length of the spacer minus 1 bp if spacer length is even or minus 1/2 bp if spacer length is odd.
  * **Sequence (5'-to-3'):** Input sequence containing the target(s) for base editing.<br>
  * **Save results as CSV file?** Type 'yes' to save the results or 'no' to proceed.<br>
    * **Alphanumeric filename:** Type an alphanumeric filename to save the results<br>
  * **New sequence?** Type 'yes' to try another sequence or 'no' to exit.<br>
 
**UNDERSTANDING THE OUTPUT:**<br>
* **Output sequence:** <br>
  * For TALEN design, the output sequence is the input sequence with the sequence of the target coordinate capitalized and highlighted in blue.
  * For FusXTBE design, the output sequence corresponds to the input sequence with all the potentially editable TC or GA motifs capitalized and highlighted in blue. Alternatively, for PTC induction, the output sequence consists of the input sequence with all targets amenable to PTC induction capitalized and highlighted in blue.<br><br>
* **Output table:**<br>
  * For TALEN design, the output table summarizes all the possible TALEN design permutations around the specified target, constrained by the design parameters. The table contains the design index of each design permutation, the target sequence of the left arm (plus a 5'T), the complementary sequence of the target sequence of the right arm (plus a 3'A), and the sequence of the spacer with the desired cut site capitalized and highlighted in blue.
  * For FusXTBE design, the output table summarizes all the possible FusXTBE design permutations around all the C-to-T base editing targets identified within the input sequence, constrained by the design parameters. The table contains the target indices, their local coordinates within the input sequence, their sequence motifs (TC or GA, or TGA, TCAA, or TCAG), the design indices, the target sequence of the left arm (plus a 3'A), the complementary sequence of the target sequence of the right arm (plus a 3'A), an exclamation mark (!) indicating potential off-target editing sites within the spacer, and the sequence of the spacer with the potential off-target sites highlighted in red and the on-target site capitalized and highlighted in blue.<br> 

**Note 1:** In FusXTBE design, there can be dozens of design permutations per target. One way to decrease output size is to define constrained custom design parameters. For example, by making the minimum and maximum lengths of all left arms, right arms, and spacers the same (e.g., using a 15-15-15 design rule), output size can be significantly decreased. It is up to the user to select the best design permutation that best fits their specific application.<br><br>
**Note 2:** For FusXTBE design, the program may ask to add nucleotides to either the 5' end, the 3' end, or both ends of the input sequence. In that case, it is possible that by adding the suggested number of nucleotides, new potential targets for C-to-T base editing are detected. One possible way around this issue is to add 'junk' extra nucleotides whenever required, such as a consecutive strecht of Ts. The user needs to consider extra targets or junk sequences when analyzing any design permutations.<br><br>
**Author information:**<br>
Santiago Restrepo-Castillo<br>
RestrepoCastillo.Santiago@mayo.edu<br>
Ph.D. Candidate | Virology and Gene Therapy<br>
Mayo Clinic Graduate School of Biomedical Sciences

In [5]:
# ----
# HEAD
# ----

# Welcome
print('-----------')
print('TALE WRITER')
print('-----------')
print('')
print("Welcome to TALE Writer! A Python-based design tool for the efficient design of TALE repeat arrays.")

# Libraries
from colorama import Back, Style
import pandas as pd
pd.set_option('display.max_colwidth', -1)
import numpy as np
import sys

# ----
# BODY
# ----

# Design TALENs or BEs?
design = 0
while design == 0:
    design = input('\nDesign:\nA. TALENs\nB. Base editors\n')
    if design.lower() == 'exit':
        sys.exit(0)
    if design not in {'A','a','B','b'}:
        design = 0
        print('Invalid input! Choose A or B.')
    else:
        design = design.lower()

# Find TC motifs or PTC sites only?
if design == 'b':
    baseeditor = 0
    while baseeditor == 0:
        baseeditor = input("\nBase editors for:\nA. 5'-TC-3' motifs within a given sequence\nB. Premature termination codon induction\n")
        if baseeditor.lower() == 'exit':
            sys.exit(0)
        if baseeditor not in {'A','a','B','b'}:
            baseeditor = 0
            print('Invalid input! Choose A or B')
        else:
            baseeditor = baseeditor.lower()

# Define design limits
default = 0
while default == 0:
    default = input('\nUse default design parameters? ')
    if default.lower() == 'exit':
        sys.exit(0)
    if default.lower() not in {'yes','y','no','n'}:
        default = 0
        print("Invalid input! Type 'yes' or 'no'")
# Default design parameters
if default.lower() in {'yes','y'}:
    leftArmMin, spacerMin, rightArmMin = 14,14,14
    leftArmMax, spacerMax, rightArmMax = 16,16,16
    window = 4
    # Print design parameters
    print('\nLeft arm minimum length (bp): 14')
    print('Left arm maximum length (bp): 16')
    print('\nSpacer minimum length (bp): 14')
    print('Spacer maximum length (bp): 16')
    print('\nRight arm minimum length (bp): 14')
    print('Right arm maximum length (bp): 16')
    print('\nTarget window: 4')
    
# Custom design parameters
else:        
    leftArmMin, leftArmMax, spacerMin, spacerMax, rightArmMin, rightArmMax, window = 0,0,0,0,0,0,-1
    # Left arm minimum length
    while leftArmMin == 0:
        leftArmMin = input('\nLeft arm minimum length (bp): ')
        if leftArmMin.lower() == 'exit':
            sys.exit(0)
        try:
            leftArmMin = int(leftArmMin)
            if leftArmMin < 10 or leftArmMin > 25:
                leftArmMin = 0
                print('Invalid input! Minimum length must be an integer between 10 and 25.')
        except:
            leftArmMin = 0
            print('Invalid input! Minimum length must be an integer between 10 and 25.')
    # Left arm maximum length
    while leftArmMax == 0:
        leftArmMax = input('\nLeft arm maximum length (bp): ')
        if leftArmMax.lower() == 'exit':
            sys.exit(0)
        try:
            leftArmMax = int(leftArmMax)
            if leftArmMax < 10 or leftArmMax > 25:
                leftArmMax = 0
                print('Invalid input! Maximum length must be an integer between 10 and 25.')
            elif leftArmMax < leftArmMin:
                leftArmMax = 0
                print('Invalid input! Maximum length cannot be less than minimum length.')
        except:
            leftArmMax = 0
            print('Invalid input! Maximum length must be an integer between 10 and 25.')
    # Spacer minimum length
    while spacerMin == 0:
        spacerMin = input('\nSpacer minimum length (bp): ')
        if spacerMin.lower() == 'exit':
            sys.exit(0)
        try:
            spacerMin = int(spacerMin)
            if spacerMin < 10 or spacerMin > 25:
                spacerMin = 0
                print('Invalid input! Minimum length must be an integer between 10 and 25.')
        except:
            spacerMin = 0
            print('Invalid input! Minimum length must be an integer between 10 and 25.')
    # Spacer maximum length
    while spacerMax == 0:
        spacerMax = input('\nSpacer maximum length (bp): ')
        if spacerMax.lower() == 'exit':
            sys.exit(0)
        try:
            spacerMax = int(spacerMax)
            if spacerMax < 10 or spacerMax > 25:
                spacerMax = 0
                print('Invalid input! Maximum length must be an integer between 10 and 25.')
            elif spacerMax < spacerMin:
                spacerMax = 0
                print('Invalid input! Maximum length cannot be less than minimum length.')
        except:
            spacerMax = 0
            print('Invalid input! Maximum length must be an integer between 10 and 25.')
    # Right arm minimum length
    while rightArmMin == 0:
        rightArmMin = input('\nRight arm minimum length (bp): ')
        if rightArmMin.lower() == 'exit':
            sys.exit(0)
        try:
            rightArmMin = int(rightArmMin)
            if rightArmMin < 10 or rightArmMin > 25:
                rightArmMin = 0
                print('Invalid input! Minimum length must be an integer between 10 and 25.')
        except:
            rightArmMin = 0
            print('Invalid input! Minimum length must be an integer between 10 and 25.')
    # Right arm maximum length
    while rightArmMax == 0:
        rightArmMax = input('\nRight arm maximum length (bp): ')
        if rightArmMax.lower() == 'exit':
            sys.exit(0)
        try:
            rightArmMax = int(rightArmMax)
            if rightArmMax < 10 or rightArmMax > 25:
                rightArmMax = 0
                print('Invalid input! Maximum length must be an integer between 10 and 25.')
            elif rightArmMax < rightArmMin:
                rightArmMax = 0
                print('Invalid input! Maximum length cannot be less than minimum length.')
        except:
            rightArmMax = 0
            print('Invalid input! Maximum length must be an integer between 10 and 25.')
    # Window
    while window == -1:
        window = input('\nTarget window: ')
        if window.lower() == 'exit':
            sys.exit(0)
        try:
            window = int(window)
            # If the spacer is even
            if (spacerMax % 2) == 0:
                if window < 0 or window > ((spacerMax-2)/2):
                    window = -1
                    print('Invalid input! The window cannot be less than 0 or greater than ' + str(int((spacerMax-2)/2)))
            # If the spacer is odd
            else:
                if window < 0 or window > ((spacerMax-1)/2):
                    window = -1
                    print('Invalid input! The window cannot be less than 0 or greater than ' + str(int((spacerMax-1)/2)))
        except:
            window = -1
            print('Invalid input! The window must be an integer.')
            
newSequence, count = True, 0
while newSequence == True:
        
    # Head of each analysis
    count += 1
    print('')
    print('-' * (13 + len(str(count))))
    print('SEQUENCE NO. ' + str(count))
    print('-' * (13 + len(str(count))))
    
    # Define input
    sequence = 0
    while sequence == 0:
        sequence = input("\nSequence (5'-to-3'): ")
        if sequence.lower() == 'exit':
            sys.exit(0)
        x, i = sequence.lower(), 0
        while i in range(0,int(len(x))):
            if x[i] not in {'a','c','g','t'}:
                print('Invalid sequence!')
                i = len(sequence)
                sequence = 0
            else:
                i += 1
    sequence = sequence.lower()
    
    # TALEN design
    if design == 'a':
        # Define the target
        coordinate, coordinates = 0, [[],[]]
        while coordinate == 0:
            coordinate = input('\nTarget coordinate: ')
            if coordinate.lower() == 'exit':
                sys.exit(0)
            try:
                coordinate = int(coordinate)
                if coordinate <= 0:
                    coordinate = 0
                    print('Invalid input! The target coordinate must be a positive integer.')
                elif coordinate > len(sequence):
                    coordinate = 0
                    print('Invalid input! The target coordinate cannot be outside of the sequence of interest.')
                else:
                    sequence = sequence[:coordinate-1] + sequence[coordinate-1:coordinate].replace(sequence[coordinate-1],sequence[coordinate-1].upper()) + sequence[coordinate:]
                    coordinates[0].append(sequence[coordinate-1])
                    coordinates[1].append(coordinate)
            except ValueError:
                coordinate = 0
                print('Invalid input! The target coordinate must be an integer.')
        # Set proceeding function at True
        proceed1 = True
    
    # Base editor design        
    else:
        # 5'-TC-3' motifs within the given sequence
        if baseeditor == 'a':
            # Find all target sites
            coordinates, tcCount, gaCount = [[],[]], 0, 0
            for i in range(0,int(len(sequence))):
                if sequence[i:i+2] == 'tc':
                    sequence = sequence[:i] + sequence[i:i+2].replace('tc','TC') + sequence[i+2:]
                    coordinates[0].append('TC')
                    coordinates[1].append(i+1)
                    tcCount += 1
                if sequence[i:i+2] == 'ga':
                    sequence = sequence[:i] + sequence[i:i+2].replace('ga','GA') + sequence[i+2:]
                    coordinates[0].append('GA')
                    coordinates[1].append(i+1)
                    gaCount += 1
            # Define proceeding function
            proceed1 = tcCount + gaCount != 0
        # Premature termination codon (PTC) induction
        else:
            # Parse sequence into triplets and find all target sites
            triplets, coordinates = [], [[],[]]
            tcaaCount, tgaCount, tcagCount = 0, 0, 0
            for i in range(0,int(len(sequence)/3)):
                triplets.append(sequence[3*i:3*i+3])
                # Turn into UAA (CAA to TAA, then UAA in mRNA [stop, both in nDNA and mtDNA])
                if triplets[i] == 'caa':
                    if i != 0:
                        if triplets[i-1][2] == 't':
                            sequence = sequence[:3*i-1] + sequence[3*i-1:3*i+3].replace('tcaa','TCAA') + sequence[3*i+3:]
                            coordinates[0].append('TCAA')
                            coordinates[1].append(3*i)
                            tcaaCount += 1
                # TGA to TAA, so UAA in mRNA (stop, both in nDNA and mtDNA)
                elif triplets[i] == 'tga':
                    sequence = sequence[:3*i] + sequence[3*i:3*i+3].replace('tga','TGA') + sequence[3*i+3:]
                    coordinates[0].append('TGA')
                    coordinates[1].append(3*i+1)
                    tgaCount += 1
                # Turn into UAG (CAG to TAG, then UAG in mRNA [stop, both in nDNA and mtDNA])
                if triplets[i] == 'cag':
                    if i != 0:
                        if triplets[i-1][2] == 't':
                            sequence = sequence[:3*i-1] + sequence[3*i-1:3*i+3].replace('tcag','TCAG') + sequence[3*i+3:]
                            coordinates[0].append('TCAG')
                            coordinates[1].append(3*i)
                            tcagCount += 1
            # Define proceeding functions                
            proceed1 = tcaaCount + tgaCount + tcagCount != 0
    
    # Verify that the input sequence is long enough
    proceed2 = True
    if proceed1: # If there are targets
        # Verify the length of the 5' end
        add5end, add3end, add5end_list, add3end_list = 0, 0, [], []
        for i in range(0,len(coordinates[0])):
            for j in range(spacerMin,spacerMax+1):
                for k in range(leftArmMin,leftArmMax+1):
                    for l in range(rightArmMin,rightArmMax+1):
                        for m in range(0,j-len(coordinates[0][i])+1): 
                            # Verify the length of the 5' end
                            if coordinates[1][i] < k+m+2:
                                proceed2, c1 = False, 0
                                while c1+coordinates[1][i] < k+m+2: c1 += 1
                                add5end_list.append(c1)
                            # Verify the length of the 3' end
                            if len(sequence[coordinates[1][i]-1+j-m:coordinates[1][i]+j-m+l]) < l+1:
                                proceed2, c2 = False, 0
                                while c2+len(sequence[coordinates[1][i]-1+j-m:coordinates[1][i]+j-m+l]) < l+1: c2 += 1
                                add3end_list.append(c2)
        # Calculate number of nucleotides that should be added 
        if add5end_list:
            add5end = np.amax(np.array(add5end_list))
            if design == baseeditor == 'b': # Base editor design for PTC induction
                    while add5end % 3 != 0: add5end += 1 # The ORF cannot be altered                         
        if add3end_list:
            add3end = np.amax(np.array(add3end_list)) 
        # Recommend adding nucleotides
        if proceed2 == False:
            # Only 5' end
            if add5end != 0 and add3end == 0:
                print("\nAdd at least " + str(add5end) + " nucleotides at the 5' end.")
            # Only 3' end
            if add5end == 0 and add3end != 0:
                print("\nAdd at least " + str(add3end) + " nucleotides at the 3' end.")
            # Both 5' and 3' end
            if add5end != 0 and add3end != 0:
                print("\nAdd at least " + str(add5end) + " nucleotides at the 5' end "\
                      "and " + str(add3end) + " nucleotides at the 3' end.")    
    else: # If there are no targets
        proceed2 = False
        print('\nThere are no target sites!')   
    
    # If the input sequence has targets and is long enough on both ends
    if proceed2:
        # Identify all possible design permutations
        design_list = [[],[],[],[],[],[],[],[]]
        for i in range(0,len(coordinates[0])): # Include all targets
            designCounter = 0
            if design == 'b':
                targetable = False # Define boolean variable to get rid of untargetable sites
            for j in range (spacerMin,spacerMax+1): # Spacer limits
                for k in range(leftArmMin,leftArmMax+1): # Left arm limits
                    for l in range(rightArmMin,rightArmMax+1): # Right arm limits
                        for m in range(0,j-len(coordinates[0][i])+1): # Slide target through spacer
                            # Left arm
                            leftArm = sequence[coordinates[1][i]-2-m-k:coordinates[1][i]-1-m].lower()
                            # Spacer (making sure that only the target is capitalized)
                            spacerX = sequence[coordinates[1][i]-1-m:coordinates[1][i]-1+j-m]
                            spacerXleft = spacerX[0:m].lower()
                            spacerXmiddle = spacerX[m:m+len(coordinates[0][i])]                            
                            spacerXright = spacerX[m+len(coordinates[0][i]):len(spacerX)].lower()
                            spacer = spacerXleft + spacerXmiddle + spacerXright     
                            # Right arm
                            rightArm = sequence[coordinates[1][i]-1+j-m:coordinates[1][i]+j-m+l].lower()
                            # Veryify that 5'T requirement is fulfilled on both ends
                            p = leftArm[0] in {'T','t'} and rightArm[len(rightArm)-1] in {'A','a'}
                            if p:
                                # If the target site is within a defined window within the spacer
                                if m + 1 >= window and m + 2 <= len(spacer) - window:                               
                                    designCounter += 1
                                    if design == 'b':
                                        targetable = True # If there is at least one viable design, the target is targetable
                                    # Start appending
                                    design_list[1].append(coordinates[1][i]) # Target coordinate
                                    design_list[2].append(coordinates[0][i]) # Target motif
                                    design_list[3].append(designCounter) # Design index
                                    design_list[4].append(leftArm) # Left arm
                                    # Spacer
                                    if design == 'a': # If designing TALENs
                                        design_list[7].append(' ') # No off-target warning
                                        spacer = spacer.replace(spacer[m], Back.BLUE + spacer[m] + Style.RESET_ALL)
                                    if design == 'b': # If designing base editors                                      
                                        if baseeditor == 'a': # If designing base editors against all 5'-TC-3' sites
                                            itsTC = sequence[coordinates[1][i]-1:coordinates[1][i]-1+2] == 'TC'
                                            itsGA = sequence[coordinates[1][i]-1:coordinates[1][i]-1+2] == 'GA'
                                            if itsTC or itsGA:
                                                spacer = spacerXleft + Back.BLUE + spacerXmiddle + Style.RESET_ALL + spacerXright
                                        if baseeditor == 'b': # If designing base editors for PTC induction
                                            itsTCAA = sequence[coordinates[1][i]-1:coordinates[1][i]-1+4] == 'TCAA'
                                            itsTGA = sequence[coordinates[1][i]-1:coordinates[1][i]-1+3] == 'TGA'
                                            itsTCAG = sequence[coordinates[1][i]-1:coordinates[1][i]-1+4] == 'TCAG'
                                            if itsTCAA or itsTGA or itsTCAG:
                                                spacer = spacerXleft + Back.BLUE + spacerXmiddle + Style.RESET_ALL + spacerXright
                                        spacer = spacer.replace('tc', Back.RED + 'tc' + Style.RESET_ALL)
                                        spacer = spacer.replace('ga', Back.RED + 'ga' + Style.RESET_ALL)
                                    design_list[5].append(spacer) # Spacer
                                    design_list[6].append(rightArm) # Right arm
                                    if 'tc' in spacer or 'ga' in spacer:
                                        design_list[7].append('!') # Off-target warning
                                    else:
                                        design_list[7].append(' ') # No off-target warning
                                                          
            # Unhighlight any untargetable targets
            if design == 'b':
                # If targetable stayed False for a specific motif, unhighlight that motif
                if targetable == False:
                    if baseeditor == 'a':
                        if sequence[coordinates[1][i]-1:coordinates[1][i]-1+2] == 'TC':
                            sequence = sequence[:coordinates[1][i]-1] + sequence[coordinates[1][i]-1:coordinates[1][i]-1+2].replace('TC','tc') + sequence[coordinates[1][i]-1+2:] 
                        if sequence[coordinates[1][i]-1:coordinates[1][i]-1+2] == 'GA':
                            sequence = sequence[:coordinates[1][i]-1] + sequence[coordinates[1][i]-1:coordinates[1][i]-1+2].replace('GA','ga') + sequence[coordinates[1][i]-1+2:]    
                    else:
                        if sequence[coordinates[1][i]-1:coordinates[1][i]-1+4] == 'TCAA':
                            sequence = sequence[:coordinates[1][i]-1] + sequence[coordinates[1][i]-1:coordinates[1][i]-1+4].replace('TCAA','tcaa') + sequence[coordinates[1][i]-1+4:] 
                        if sequence[coordinates[1][i]-1:coordinates[1][i]-1+3] == 'TGA':
                            sequence = sequence[:coordinates[1][i]-1] + sequence[coordinates[1][i]-1:coordinates[1][i]-1+3].replace('TGA','tga') + sequence[coordinates[1][i]-1+3:] 
                        if sequence[coordinates[1][i]-1:coordinates[1][i]-1+4] == 'TCAG':
                            sequence = sequence[:coordinates[1][i]-1] + sequence[coordinates[1][i]-1:coordinates[1][i]-1+4].replace('TCAG','tcag') + sequence[coordinates[1][i]-1+4:] 
                                              
        # Correctly append the target numbers
        targetCounter = 1
        for i in range(0,len(design_list[1])):
            if i > 0:
                if design_list[1][i] != design_list[1][i-1]:
                    targetCounter += 1
            design_list[0].append(targetCounter)
        
        # If there are no viable design permutations
        if not design_list[0]:
            print('\nThere are no viable design permutations!')
            # Ask to try another sequence
            newSequenceX = 0
            while newSequenceX == 0:
                newSequenceX = input("\nNew sequence? ")
                if newSequenceX.lower() == 'exit':
                    sys.exit(0)
                newSequenceX = newSequenceX.lower()
                if newSequenceX not in {'yes','y','no','n'}:
                    newSequenceX = 0
                    print("Invalid input! Type 'yes' or 'no'")
                else:
                    if newSequenceX == 'no':
                        newSequence = False
                        input('\nBye :-)')
        # If there are viable design permutations
        else:            
            # Highlight the target sites
            sequenceX = sequence # Define auxiliar variable to allow troubleshooting with original sequence variable
            if design == 'a':
                sequenceX = sequenceX.replace(sequenceX[coordinate-1], Back.BLUE + sequenceX[coordinate-1] + Style.RESET_ALL)
                print('\nSequence (with target site): ' + sequenceX)
            else:
                if baseeditor == 'a':
                    sequenceX = sequenceX.replace('TC', Back.BLUE + 'TC' + Style.RESET_ALL)
                    sequenceX = sequenceX.replace('GA', Back.BLUE + 'GA' + Style.RESET_ALL)
                else:
                    sequenceX = sequenceX.replace('TCAA', Back.BLUE + 'TCAA' + Style.RESET_ALL)
                    sequenceX = sequenceX.replace('TGA', Back.BLUE + 'TGA' + Style.RESET_ALL)
                    sequenceX = sequenceX.replace('TCAG', Back.BLUE + 'TCAG' + Style.RESET_ALL)
                print('\nSequence (with target sites): ' + sequenceX)
                       
            # Display results
            print('\nResults:\n')   
            # Define a blank for displaying purposes
            counterList = []
            for i in range(0, len(design_list[5])):
                counterList.append(design_list[5][i].count('tc') + design_list[5][i].count('ga'))
            blank = '         '
            # Create a dataframe containing the results
            if design == 'a': # If designing TALENs
                results_df = pd.DataFrame({
                            'Design':design_list[3],
                            "5'T + Left arm":design_list[4],
                            "Right arm + 3'A":design_list[6],
                            'Spacer' + blank:design_list[5]})
            if design == 'b': # If designing base editors
                results_df = pd.DataFrame({
                            'Target':design_list[0],
                            'Coordinate':design_list[1],
                            'Motif':design_list[2],
                            'Design':design_list[3],
                            "5'T + Left arm":design_list[4],
                            "Right arm + 3'A":design_list[6],
                            '!':design_list[7],
                            'Spacer' + (max(counterList)+1)*blank:design_list[5]})
            results_df.index += 1
            print(results_df.to_string(index=False))
            
            # Save results
            save, name = 0, 0
            while save == 0:
                save = input("\nSave results as CSV file? ")
                if save.lower() == 'exit':
                    sys.exit(0)
                save = save.lower()
                if save not in {'yes','y','no','n'}:
                    save = 0
                    print("Invalid input! Type 'yes' or 'no'")
                else:
                    if save == 'yes':
                        while name == 0:
                            name = input('\nAlphanumeric filename: ')
                            if name.lower() == 'exit':
                                sys.exit(0)
                            if name.isalnum():
                                # Get rid of colorama markers
                                for i in range(0, len(design_list[5])):
                                    design_list[5][i] = design_list[5][i].replace('\x1b[41m','')
                                    design_list[5][i] = design_list[5][i].replace('\x1b[44m','')
                                    design_list[5][i] = design_list[5][i].replace('\x1b[0m','')
                                # Create a new dataframe with clean information (no highlights)
                                if design == 'a': # If designing TALENs
                                    save_df = pd.DataFrame({
                                                'Design':design_list[3],
                                                "5'T + Left arm":design_list[4],
                                                'Spacer':design_list[5],
                                                "Right arm + 3'A":design_list[6]})
                                if design == 'b': # If designing base editors
                                    save_df = pd.DataFrame({
                                            'Target':design_list[0],
                                            'Coordinate':design_list[1],
                                            'Motif':design_list[2],
                                            'Design':design_list[3],
                                            "5'T + Left arm":design_list[4],
                                            'Spacer':design_list[5],
                                            "Right arm + 3'A":design_list[6],
                                            '!':design_list[7]})
                                save_df.index += 1
                                save_df.to_csv(name + '.csv') # Save the dataframe
                            else:
                                name = 0
                                print('Invalid input! File name must be alphanumeric')
                            
            # Even if everything worked, ask to try another sequence
            newSequenceX = 0
            while newSequenceX == 0:
                newSequenceX = input('\nNew sequence? ')
                if newSequenceX.lower() == 'exit':
                    sys.exit(0)
                newSequenceX = newSequenceX.lower()
                if newSequenceX not in {'yes','y','no','n'}:
                    newSequenceX = 0
                    print("Invalid input! Type 'yes' or 'no'")
                else:
                    if newSequenceX == 'no':
                        newSequence = False
                        input('\nBye :-)')
            
    # If the input sequence doesn't have any targets or isn't long enough, ask to try another sequence
    else:
        newSequenceX = 0
        while newSequenceX == 0:
            newSequenceX = input('\nNew sequence? ')
            if newSequenceX.lower() == 'exit':
                sys.exit(0)
            newSequenceX = newSequenceX.lower()
            if newSequenceX not in {'yes','y','no','n'}:
                newSequenceX = 0
                print("Invalid input! Type 'yes' or 'no'")
            else:
                if newSequenceX == 'no':
                    newSequence = False
                    input('\nBye :-)')

-----------
TALE WRITER
-----------

Welcome to TALE Writer! A Python-based design tool for the efficient design of TALE repeat arrays.

Design:
A. TALENs
B. Base editors
B

Base editors for:
A. 5'-TC-3' motifs within a given sequence
B. Premature termination codon induction
a

Use default design parameters? yes

Left arm minimum length (bp): 14
Left arm maximum length (bp): 16

Spacer minimum length (bp): 14
Spacer maximum length (bp): 16

Right arm minimum length (bp): 14
Right arm maximum length (bp): 16

Target window: 4

--------------
SEQUENCE NO. 1
--------------

Sequence (5'-to-3'): tttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttttattaatcccctggcccaacccgtcatctactctaccatctttgcaggcacactcatcacagcgctaagctcgcactgattttttacctgagtaggcctagaaataaacatgctagcttttattccagttctaaccaaaaaaataaaccctcgttccacagaagctgccatcaagtatttcctcacgcaagcaaccgcatccataatccttctaatagctatcctcttcaacaatatactctccggacaatgaaccataaccaatactaccaatcaatactcatcattaataatcataatagctatagcaataaaactaggaatagccccctttcactt


Save results as CSV file? no

New sequence? no

Bye :-)
